# 02a - Ethereum Fraud Detection baselines

Random Forest and LightGBM on the Ethereum Fraud Detection dataset
(9,841 addresses, 45 features). Stratified 60/20/20 split, standardized
features, class-balanced training. The optimal decision threshold is
picked on the validation split and reused for the test split.

**Outputs:**

- `models/saved/ethereum_rf.joblib` / `ethereum_lgb.joblib`
- `models/saved/ethereum_scaler.joblib`
- `data/processed/ethereum_splits.npz`
- `experiments/results/ethereum_feature_names.npy`
- `experiments/results/ethereum_baselines_results.csv`
- `experiments/figures/ethereum_{baselines,feature_importance}.png`


In [ ]:
import os
from datetime import datetime

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

from xai_blockchain_framework.config import PATHS, set_seed
from xai_blockchain_framework.models import compute_metrics, find_optimal_threshold

set_seed(42)

DATA_PATH = str(PATHS.data_raw)
FIGURES_PATH = str(PATHS.figures_dir)
RESULTS_PATH = str(PATHS.results_dir)
MODELS_PATH = str(PATHS.models_dir)
PROCESSED_PATH = str(PATHS.data_processed)
for p in [FIGURES_PATH, RESULTS_PATH, MODELS_PATH, PROCESSED_PATH]:
    os.makedirs(p, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'rf': '#2ecc71', 'lgb': '#9b59b6'}

print('02a - Ethereum baselines')
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")


## 1. Load and preprocess


In [ ]:
df = pd.read_csv(os.path.join(DATA_PATH, 'Ethereum_Fraud_Detection.csv'))
print(f'Loaded {len(df)} addresses with {len(df.columns)} columns')

df = df.drop(columns=[c for c in ['Unnamed: 0', 'Index', 'Address'] if c in df.columns])
y = df['FLAG'].values
X_df = df.drop(columns=['FLAG'])

cat_cols = X_df.select_dtypes(include=['object']).columns.tolist()
if cat_cols:
    print(f'Dropping non-numeric columns: {cat_cols}')
    X_df = X_df.drop(columns=cat_cols)

X_df = X_df.fillna(X_df.median())
feature_names = X_df.columns.tolist()
X = X_df.values.astype(np.float32)
print(f'Shape: {X.shape}, fraud ratio: {y.mean():.4f}')
np.save(os.path.join(RESULTS_PATH, 'ethereum_feature_names.npy'), feature_names)


## 2. Stratified split (60/20/20), StandardScaler


In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42)

scaler = StandardScaler()
X_train_n = scaler.fit_transform(X_train)
X_val_n = scaler.transform(X_val)
X_test_n = scaler.transform(X_test)
joblib.dump(scaler, os.path.join(MODELS_PATH, 'ethereum_scaler.joblib'))

scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f'Train: {len(y_train)} ({int(y_train.sum())} fraud)  '
      f'Val: {len(y_val)} ({int(y_val.sum())} fraud)  '
      f'Test: {len(y_test)} ({int(y_test.sum())} fraud)')
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

np.savez(
    os.path.join(PROCESSED_PATH, 'ethereum_splits.npz'),
    X_train=X_train_n, X_val=X_val_n, X_test=X_test_n,
    y_train=y_train, y_val=y_val, y_test=y_test,
)


## 3. Train Random Forest and LightGBM


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=15, min_samples_leaf=2,
    class_weight='balanced', random_state=42, n_jobs=-1,
).fit(X_train_n, y_train)
rf_proba_val = rf.predict_proba(X_val_n)[:, 1]
rf_proba_test = rf.predict_proba(X_test_n)[:, 1]
rf_threshold, _ = find_optimal_threshold(y_val, rf_proba_val)
rf_val = compute_metrics(y_val, rf_proba_val, rf_threshold)
rf_test = compute_metrics(y_test, rf_proba_test, rf_threshold)
print(f"RF threshold={rf_threshold:.3f}  val F1={rf_val['F1']:.4f}  test F1={rf_test['F1']:.4f}")
joblib.dump(rf, os.path.join(MODELS_PATH, 'ethereum_rf.joblib'))

lgbm = lgb.LGBMClassifier(
    n_estimators=200, max_depth=7, learning_rate=0.05,
    scale_pos_weight=scale_pos_weight, random_state=42, verbose=-1,
).fit(X_train_n, y_train)
lgb_proba_val = lgbm.predict_proba(X_val_n)[:, 1]
lgb_proba_test = lgbm.predict_proba(X_test_n)[:, 1]
lgb_threshold, _ = find_optimal_threshold(y_val, lgb_proba_val)
lgb_val = compute_metrics(y_val, lgb_proba_val, lgb_threshold)
lgb_test = compute_metrics(y_test, lgb_proba_test, lgb_threshold)
print(f"LGB threshold={lgb_threshold:.3f}  val F1={lgb_val['F1']:.4f}  test F1={lgb_test['F1']:.4f}")
joblib.dump(lgbm, os.path.join(MODELS_PATH, 'ethereum_lgb.joblib'))


## 4. Summary table and feature importance plot


In [ ]:
results = [
    {'Dataset': 'Ethereum', 'Model': 'Random Forest', 'Split': 'Val',  **rf_val},
    {'Dataset': 'Ethereum', 'Model': 'Random Forest', 'Split': 'Test', **rf_test},
    {'Dataset': 'Ethereum', 'Model': 'LightGBM',      'Split': 'Val',  **lgb_val},
    {'Dataset': 'Ethereum', 'Model': 'LightGBM',      'Split': 'Test', **lgb_test},
]
results_df = pd.DataFrame(results)
print(results_df[['Model', 'Split', 'F1', 'Precision', 'Recall', 'ROC-AUC']].to_string(index=False))
results_df.to_csv(os.path.join(RESULTS_PATH, 'ethereum_baselines_results.csv'), index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
metrics_plot = ['F1', 'Recall', 'Precision', 'ROC-AUC']
x = np.arange(len(metrics_plot)); width = 0.35
for ax, (split_name, rf_m, lgb_m) in zip(
    axes, [('Validation', rf_val, lgb_val), ('Test', rf_test, lgb_test)],
):
    rf_vals = [rf_m[m] for m in metrics_plot]
    lgb_vals = [lgb_m[m] for m in metrics_plot]
    ax.bar(x - width / 2, rf_vals, width, label='Random Forest', color=COLORS['rf'])
    ax.bar(x + width / 2, lgb_vals, width, label='LightGBM', color=COLORS['lgb'])
    ax.set_xticks(x); ax.set_xticklabels(metrics_plot)
    ax.set_ylabel('Score'); ax.set_title(f'Ethereum - {split_name}')
    ax.set_ylim(0, 1.15); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'ethereum_baselines.png'), dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (model_obj, name, color) in zip(
    axes, [(rf, 'Random Forest', COLORS['rf']), (lgbm, 'LightGBM', COLORS['lgb'])],
):
    imp = pd.DataFrame({'feature': feature_names, 'importance': model_obj.feature_importances_})
    imp = imp.sort_values('importance', ascending=False).head(15)
    ax.barh(imp['feature'], imp['importance'], color=color)
    ax.set_xlabel('Importance'); ax.set_title(f'{name}  top 15')
    ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'ethereum_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print('\nSaved: ethereum_rf.joblib, ethereum_lgb.joblib, ethereum_scaler.joblib, '
      'ethereum_splits.npz, ethereum_feature_names.npy, '
      'ethereum_baselines_results.csv, ethereum_baselines.png, ethereum_feature_importance.png')
